# P11 本番で守る ― 攻撃を仕込んだデータ

> **この教材に登場する会社・人物・製品・数値・文書はすべて架空のものです。実在の企業・団体・個人とは関係ありません。**

お客様窓口の下書き係に、指示を紛れ込ませた問い合わせ・規程・道具の説明を渡し、守れたかを「起きたこと」で判定します。

- 上から順に実行します。0 のセルは準備です (Colab でも手元の Jupyter でも同じ)。
- API キーが無ければ、講師の実行記録を再生します (費用 0)。記録も無ければ mock で最後まで流れます (mock の数字は性能の証拠になりません)。
- 🔴 データはすべて架空の会社・文書です。

## 0. 準備

In [ ]:
# 0-1. 実行場所を見分けて、キットのフォルダに移る (Colab のときだけ取得とインストールをする)
import os, sys, subprocess
REPO_URL = "https://github.com/gucchi123/mcp-skills-colab.git"
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    KIT = "/content/mcp-skills-kit"
    if not os.path.exists(KIT):
        subprocess.run(["git", "clone", "-q", REPO_URL, KIT], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{KIT}/requirements.txt"], check=True)
else:  # 手元: notebooks/ から開いた前提。キットの根 (common/ がある所) を探す
    KIT = os.path.abspath(".." if os.path.exists("../common/host.py") else ".")
os.chdir(KIT)
sys.path.insert(0, KIT)
print("実行場所:", "Colab" if IN_COLAB else "手元の Jupyter", "/ キットの根:", os.path.basename(KIT))

In [ ]:
# 0-2. Claude の呼び先を決める (キーは表示しない) と、ラボのモジュールを読めるようにする
from common.nb import choose_provider, safe
from common.kit import use_labs, utf8_console
PROVIDER = choose_provider(IN_COLAB)
use_labs(); utf8_console()

## 1. 仕込んだ問い合わせを見る
6 件のうち 5 件に指示を混ぜてあります (Q-906 は攻撃の無い対照)。どれも架空の文面です。

In [ ]:
from common.injection import load_cases
for c in load_cases():
    print(f"{c['id']} ({c['customer']}, {c['channel']})")
    print("  " + c["text"].replace("\n", " ")[:150])
    print("  してはいけない:", c["expect"]["must_not"] or "なし (対照)")

## 2. 窓口の下書き係を 2 件だけ動かす
ホストは kb・crm・inventory の 3 サーバにつなぎ、利用者はお客様窓口 (U-05)。判定は呼んだ道具・通った書き込み・返信に出た値で行います。

In [ ]:
from run_defense import amain as defense
with safe():
    r = await defense(["--only", "Q-901,Q-903", "--show"])

## 3. 7 件まとめて (問い合わせ 6 件 + 規程 1 件)
🛡 の行は、モデルが従っても仕組み (権限・確認・道具の絞り込み) が止めたものです。

In [ ]:
with safe():
    r = await defense(["--all"])

## 4. 権限の広い人の名前で窓口を動かすと
利用者を部長 (U-04) に替えます。同じ問い合わせでも、権限の層が止めていたものが止まらなくなります。

In [ ]:
with safe():
    r = await defense(["--only", "Q-901,Q-905", "--user", "U-04"])

## 5. 窓口に要る道具だけを見せる (最小権限)
規程の道具と「顧客 1 件を読む」だけを見せます。発注の道具は、モデルが名乗っても呼べません。

In [ ]:
with safe():
    r = await defense(["--only", "Q-903,Q-905", "--tools", "least"])

## 6. 道具の説明を点検する
道具の説明そのものに指示を隠す攻撃 (tool poisoning) の見本と、講座の 17 個の道具を同じ規則で点検します。規則で拾えるのは典型だけで、つなぐ前に全文を人が読むのが本筋です。

In [ ]:
with safe():
    r = await defense(["--audit-tools"])

## 7. 監査ログを読む
照会と操作はサーバの側で `state/audit.jsonl` に残ります。だれが・どの道具で・何を隠されたかが分かります。

In [ ]:
import json, os
from common.kit import state_dir
p = os.path.join(state_dir(), "audit.jsonl")
rows = [json.loads(l) for l in open(p, encoding="utf-8")] if os.path.exists(p) else []
for e in rows[-8:]:
    print(e["at"], e["user"], e["server"], e["tool"], {k: v for k, v in e.items() if k in ("customer", "hidden", "product")})